# Evidence Quest：Mission Control

欢迎接案。你不是来背诵 RAG 术语的，而是要为一个真实的人做出一台能查证资料的证据工作台。

本 Notebook 是整个项目的任务控制台：先选择案件主题、服务对象和必须拒答的边界，再把它保存成后续所有阶段共同使用的项目合同。

## Evidence Quest 任务卡：Mission Control：接手第一宗案件

**你的身份：** 知识库调查员实习生  
**案件背景：** 案件资料散落在 Markdown、PDF 和实验记录里。你的第一项任务是建立调查地图，知道每条证据从哪里来、要交给谁。

### 本关专业 Goal

建立一份可复现的项目任务档案，并说清四阶段如何组成一条产品链路。

### 你要交付的作品

**项目任务地图 + 第一份案件档案**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：案件接收员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 先决定作品给谁用

请选择一个你愿意持续追问的问题领域，例如：课程资料、游戏世界观、旅行攻略、开源项目文档或自己的学习笔记。主题不需要宏大，但必须能让真实用户提出至少 5 个问题。

**重要原则：** 用户问题和‘不知道’边界先于模型。没有这两项，后面的检索指标和引用都不知道在保护什么。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase0'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase0
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 定义几个可直接开始的案件主题；学习者也可以在后面替换成自己的主题。
case_options = [
    {"name": "校园知识库失踪案", "audience": "需要复习课程资料的同学", "question": "这条知识来自哪份资料？", "refusal": "资料里没有证据时必须说不知道"},
    {"name": "游戏世界观考据案", "audience": "想核对设定的玩家", "question": "这个角色或事件在原文哪一段？", "refusal": "不能把猜测当成官方设定"},
    {"name": "个人学习资料侦探案", "audience": "希望快速回顾笔记的自己", "question": "我以前在哪个文件记录过这个概念？", "refusal": "找不到原文时必须返回待补证据"},
]

# 选择一个案件编号；先用 1 跑通，再改成 2 或 3 观察整个项目的主题变化。
selected_case_number = 1

# 检查编号是否落在选项范围内，尽早发现拼写或输入错误。
assert 1 <= selected_case_number <= len(case_options)

# 根据人类可读的编号取出案件配置。
selected_case = case_options[selected_case_number - 1]

# 写下项目必须遵守的用户故事和拒答边界。
quest_profile = {
    "case_name": selected_case["name"],
    "audience": selected_case["audience"],
    "must_answer": selected_case["question"],
    "must_refuse": selected_case["refusal"],
    "questions": [
        selected_case["question"],
        "哪些词能准确找到证据？",
        "不同切分方式会不会漏掉线索？",
        "搜索变快后质量有没有下降？",
        "用户能不能点击引用回到原文？",
    ],
    "xp": 0,
    "badges": ["案件接收员"],
}

# 创建处理数据目录，确保第一次运行也能保存档案。
profile_directory = ROOT / "data" / "processed"
profile_directory.mkdir(parents=True, exist_ok=True)

# 定义后续 Notebook 读取的统一项目合同路径。
profile_path = profile_directory / "evidence_quest_profile.json"

# 以 UTF-8 保存中文案件档案，缩进让它也能作为作品展示。
profile_path.write_text(json.dumps(quest_profile, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印案件合同，让学习者确认自己不是在为抽象的 demo 学习。
print("已接手案件:", quest_profile["case_name"])
print("服务对象:", quest_profile["audience"])
print("必须回答:", quest_profile["must_answer"])
print("必须拒答:", quest_profile["must_refuse"])
print("已保存:", profile_path)

# 只有关键字段齐全，后面的阶段才允许开始。
assert {"case_name", "audience", "must_answer", "must_refuse", "questions"} <= quest_profile.keys()

已接手案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
必须回答: 这条知识来自哪份资料？
必须拒答: 资料里没有证据时必须说不知道
已保存: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\evidence_quest_profile.json


## 你的第一张案件卡

请在下方 Markdown 中写下：

- 真实观众是谁；
- 观众会问的 5 个问题；
- 哪一种回答必须回到原文；
- 哪一种情况必须明确说“不知道”。

这张卡会直接影响后面的 qrels、Boss Query 和 API 验收。它不是装饰性的产品文案，而是测试标准的来源。

In [4]:
# 读取刚刚保存的案件档案，确认跨 Notebook 的项目合同可复用。
saved_profile = json.loads(profile_path.read_text(encoding="utf-8"))

# 打印五道挑战题，后面的检索评估会逐题使用它们。
for question_number, question in enumerate(saved_profile["questions"], start=1):
    print(f"Query {question_number}: {question}")

# 交付标准是问题清单足够驱动后续实验，而不是只存在一个标题。
assert len(saved_profile["questions"]) >= 5

Query 1: 这条知识来自哪份资料？
Query 2: 哪些词能准确找到证据？
Query 3: 不同切分方式会不会漏掉线索？
Query 4: 搜索变快后质量有没有下降？
Query 5: 用户能不能点击引用回到原文？


## Mission Control 通关

现在你已经有了一个可展示的项目起点：一份为真实观众设计的案件合同。接下来每个 Phase 都会给这宗案件增加新的能力，最后合并成 Evidence Desk。

## Boss Challenge：把案件的服务对象和‘必须说不知道’条件改成你自己的真实主题，并重新生成档案。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [5]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [6]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/project_orientation.json', 'data/processed/evidence_quest_profile.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\project_orientation.json', 'data\\processed\\evidence_quest_profile.json']
